In [ ]:
from pathlib import Path
from stable_baselines3 import SAC
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import (
    DummyVecEnv, VecMonitor, VecNormalize, VecFrameStack, VecCheckNan, VecTransposeImage
)
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback, CallbackList

SEED = 42
run_dir = Path("experiments/sac_carracing")
run_dir.mkdir(parents=True, exist_ok=True)

# Train env
train_env = make_vec_env(
    "CarRacing-v3",
    n_envs=4,
    seed=SEED,
    vec_env_cls=DummyVecEnv,
    monitor_dir=str(run_dir / "monitor_train"),
)
train_env = VecMonitor(train_env)
train_env = VecFrameStack(train_env, n_stack=4)
train_env = VecTransposeImage(train_env)
train_env = VecNormalize(train_env, norm_obs=False, norm_reward=True, clip_reward=10.0)

# Eval env (separate)
eval_env = make_vec_env(
    "CarRacing-v3",
    n_envs=1,
    seed=SEED + 1000,
    vec_env_cls=DummyVecEnv,
    monitor_dir=str(run_dir / "monitor_eval"),
)
eval_env = VecMonitor(eval_env)
eval_env = VecFrameStack(eval_env, n_stack=4)
eval_env = VecTransposeImage(eval_env)
eval_env = VecNormalize(eval_env, norm_obs=False, norm_reward=False, training=False)

eval_cb = EvalCallback(
    eval_env=eval_env,
    best_model_save_path=str(run_dir / "best_model"),
    log_path=str(run_dir / "eval_logs"),
    eval_freq=10_000,
    n_eval_episodes=5,
    deterministic=True,
)

ckpt_cb = CheckpointCallback(
    save_freq=25_000,
    save_path=str(run_dir / "checkpoints"),
    name_prefix="sac",
)

model = SAC(
    "CnnPolicy",
    train_env,
    seed=SEED,
    device="mps",
    verbose=1,
    tensorboard_log=str(run_dir / "tb"),
)

model.learn(total_timesteps=100_000, callback=CallbackList([eval_cb, ckpt_cb]))
model.save(str(run_dir / "final_model"))
train_env.save(str(run_dir / "vecnormalize.pkl"))

objc[22284]: Class SDLApplication is implemented in both /Users/nveshaan/Developer/marl_f1/.venv/lib/python3.12/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x11ede0890) and /Users/nveshaan/Developer/marl_f1/.venv/lib/python3.12/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x12d7a92c8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[22284]: Class SDLAppDelegate is implemented in both /Users/nveshaan/Developer/marl_f1/.venv/lib/python3.12/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x11ede08e0) and /Users/nveshaan/Developer/marl_f1/.venv/lib/python3.12/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x12d7a9318). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[22284]: Class SDLTranslatorResponder is implemented in both /Users/nveshaan/Developer/marl_f1/.venv/lib/python3.12/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x11ede0958) 

Using mps device
Logging to experiments/sac_carracing/tb/SAC_4
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1e+03    |
|    ep_rew_mean     | -38.1    |
| time/              |          |
|    episodes        | 4        |
|    fps             | 27       |
|    time_elapsed    | 144      |
|    total_timesteps | 4000     |
| train/             |          |
|    actor_loss      | -8.29    |
|    critic_loss     | 0.0871   |
|    ent_coef        | 0.747    |
|    ent_coef_loss   | -1.47    |
|    learning_rate   | 0.0003   |
|    n_updates       | 974      |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1e+03    |
|    ep_rew_mean     | -35.5    |
| time/              |          |
|    episodes        | 8        |
|    fps             | 27       |
|    time_elapsed    | 295      |
|    total_timesteps | 8000     |
| train/             |          |
|    actor_loss    

In [ ]:
vec_env = model.get_env()
obs = vec_env.reset()
for i in range(1000):
    action, _state = model.predict(obs, deterministic=True)
    obs, reward, done, info = vec_env.step(action)
    vec_env.render("human")

In [ ]:
from pathlib import Path
from stable_baselines3 import SAC
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import DummyVecEnv, VecMonitor, VecNormalize, VecFrameStack

run_dir = Path("experiments/sac_carracing")

eval_env = make_vec_env("CarRacing-v3", n_envs=1, vec_env_cls=DummyVecEnv)
eval_env = VecMonitor(eval_env)
eval_env = VecNormalize.load(str(run_dir / "vecnormalize.pkl"), eval_env)
eval_env.training = False
eval_env.norm_reward = False
eval_env = VecFrameStack(eval_env, n_stack=4)

model = SAC.load(str(run_dir / "best_model" / "best_model.zip"), env=eval_env, device="mps")
mean_r, std_r = evaluate_policy(model, eval_env, n_eval_episodes=20, deterministic=True)
print(mean_r, std_r)